# Evaluation Loop

Runs every active model on every active feature track, target, horizon and pseudo-out-of-sample fold, then saves per-run metrics, predictions and runtimes to `results/raw/<model>/<track>.parquet`.

This notebook only holds configuration. The logic lives in `src/`:

- `src/evaluation/` — folds, track preparation, metrics, the loop itself
- `src/models/` — model functions and `MODEL_REGISTRY`
- `src/features/quantum/` — quantum / classical reservoir feature generators (precomputed into tracks by `scripts/build_quantum_tracks.py`)

To add a model, see the docstring in `src/models/__init__.py`. To add a track, add an entry to `TRACK_CONFIG` below.

## 1. Environment setup

In [ ]:
import os
import sys
from pathlib import Path

COLAB = False  # set to True when running in Google Colab

if COLAB:
    REPO_URL  = "https://github.com/manuelarceaguirre/capstone-quantum.git"
    BRANCH    = "clean"
    REPO_ROOT = Path("/content/capstone-quantum")

    %cd /content
    !rm -rf capstone-quantum
    !git clone -b {BRANCH} {REPO_URL}
    %cd /content/capstone-quantum
    !pip install -q pmdarima
    sys.path.insert(0, str(REPO_ROOT))
else:
    # local: walk up from the working directory to the repo root;
    # src/ is importable via `pip install -e .` (see environment.yml)
    REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

print("Repo root:", REPO_ROOT)

In [ ]:
import warnings

import pandas as pd

from src.evaluation import (
    disaggregate_by_usrec,
    explode_predictions,
    generate_folds,
    load_results,
    load_tracks,
    run_evaluation,
    shared_track,
    summarize,
    write_csv,
)
from src.models import MODEL_REGISTRY

warnings.filterwarnings("ignore")

## 2. Configuration

In [ ]:
# --- Paths ---
DATA_DIR    = REPO_ROOT / "data" / "processed"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# --- Window parameters ---
WINDOW_MONTHS = 120   # training window length
TEST_MONTHS   = 1     # test window length
STEP_MONTHS   = 1     # fold step size
MAX_FOLDS     = None  # set to a small int for a quick smoke test
RUN_BY        = None  # your name, recorded on every result row (default: OS user)
RESUME        = False # True: skip runs already saved in results/ (e.g. after a crash or
                      # Colab disconnect). Keep False after changing a model's code.

# --- What to run ---
HORIZON_LIST  = [1, 3, 6]
TARGET_LIST   = ["INDPRO", "PAYEMS", "CPIAUCSL", "S&P 500"]
ACTIVE_TRACKS = ["B"]                              # any keys of TRACK_CONFIG
ACTIVE_MODELS = ["random_walk", "AR6", "arima"]    # any keys of MODEL_REGISTRY

# --- Track registry: TRACK_CONFIG[track][target] = {"type", "path"} ---
TRACK_CONFIG = {
    "A":      shared_track("dataframe", DATA_DIR / "track_A_full.parquet", TARGET_LIST),
    "B":      {t: {"type": "dataframe", "path": DATA_DIR / f"track_b_{t}.parquet"} for t in TARGET_LIST},
    "C":      shared_track("pipeline", DATA_DIR / "track_C_pipe.pkl", TARGET_LIST),       # LASSO, tabled
    "D":      shared_track("pipeline", DATA_DIR / "track_D_pipe.pkl", TARGET_LIST),       # PCA ~30 factors
    "D_mini": shared_track("pipeline", DATA_DIR / "track_D_mini_pipe.pkl", TARGET_LIST),  # PCA 15 factors
    # reservoir tracks: built by scripts/build_quantum_tracks.py
    "QRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_QRC_{t}.parquet"} for t in TARGET_LIST},
    "CRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_CRC_{t}.parquet"} for t in TARGET_LIST},
}

MODELS = {name: MODEL_REGISTRY[name] for name in ACTIVE_MODELS}

print(f"Window:   {WINDOW_MONTHS}m train / {TEST_MONTHS}m test / {STEP_MONTHS}m step")
print(f"Targets:  {TARGET_LIST}")
print(f"Horizons: {HORIZON_LIST}")
print(f"Tracks:   {ACTIVE_TRACKS}")
print(f"Models:   {ACTIVE_MODELS}")

## 3. Load data and build folds

In [ ]:
track_data       = load_tracks(TRACK_CONFIG, ACTIVE_TRACKS, TARGET_LIST)
stationary_panel = pd.read_parquet(DATA_DIR / "stationary_panel.parquet")
targets          = pd.read_parquet(DATA_DIR / "targets.parquet")
usrec            = pd.read_parquet(DATA_DIR / "usrec.parquet")

print(f"Stationary panel {stationary_panel.shape}  {stationary_panel.index.min().date()} -> {stationary_panel.index.max().date()}")
print(f"Targets          {targets.shape}")

In [ ]:
# all feature panels share the stationary panel's monthly index
folds = generate_folds(stationary_panel.index, WINDOW_MONTHS, TEST_MONTHS, STEP_MONTHS)
if MAX_FOLDS is not None:
    folds = folds[:MAX_FOLDS]

print(f"Total folds: {len(folds)}")
print(f"  first: train {folds[0]['train_start'].date()} -> {folds[0]['train_end'].date()}, test {folds[0]['test_start'].date()}")
print(f"  last:  train {folds[-1]['train_start'].date()} -> {folds[-1]['train_end'].date()}, test {folds[-1]['test_start'].date()}")

## 4. Run

Each track's results are saved to `results/raw/<model>/<track>.parquet` every 25 folds and when that track finishes. After a crash or disconnect, set `RESUME = True` and rerun: runs already saved are skipped.

In [ ]:
results_df = run_evaluation(
    folds            = folds,
    tracks           = ACTIVE_TRACKS,
    targets          = TARGET_LIST,
    horizons         = HORIZON_LIST,
    models           = MODELS,
    targets_df       = targets,
    track_config     = TRACK_CONFIG,
    track_data       = track_data,
    stationary_panel = stationary_panel,
    run_by           = RUN_BY,
    results_dir      = RESULTS_DIR,  # checkpoint every 25 folds and at the end of each track
    resume           = RESUME,
)

n_err = results_df["error"].notna().sum()
print(f"Runs with errors: {n_err}")
if n_err:
    print(results_df.loc[results_df["error"].notna(), ["model", "track", "target", "error"]].drop_duplicates("error").head())

## 5. Summarize

Saving merges rather than overwrites: only the rows a run recomputed are replaced, so teammates running other models or tracks are unaffected.

In [ ]:
# summaries cover everything on disk for the active models/tracks, not just this run
all_results = load_results(RESULTS_DIR, models=ACTIVE_MODELS, tracks=ACTIVE_TRACKS)
summary = summarize(all_results)
summary

In [ ]:
# recession (1) vs expansion (0)
summary_disagg = disaggregate_by_usrec(load_results(RESULTS_DIR), usrec)  # all models/tracks on disk
write_csv(summary_disagg, RESULTS_DIR / "summary_disaggregated.csv")
summary_disagg

In [ ]:
# one row per (run, test date): y_true vs y_pred
predictions = explode_predictions(all_results)
runtime = all_results.groupby(["model", "track"])["runtime_sec"].agg(["mean", "sum"]).round(3)
display(predictions.head(), runtime)